**Run with:** `pixi run -e notebooks jupyter lab notebooks/drafts/modis_composite_comparison.ipynb`

---

# MODIS Composite Comparison — picking the right MCDWD composite for Atlantis

**Question:** which MCDWD flood composite — **F1C** (1-day, cloud-shadow
screened), **F2** (2-day max), or **F3** (3-day max) — should Atlantis use as
its MODIS default?

Per the [MCDWD User Guide](https://www.earthdata.nasa.gov/s3fs-public/2025-12/MCDWD_VCDWD_UserGuide_RevF.pdf),
longer composites suppress cloud-shadow false positives at the cost of
temporal sharpness. This notebook makes that trade-off **visible and
quantified**, across:

1. **Both resolutions** — native 250 m `processed/` vs. harmonised 1-arcmin
   `harmonised/` rasters, so we see what nearest-neighbour resampling does to
   each composite's codes.
2. **All five example flood events** — Valencia, Harvey, Bihar, Vamco and West
   Africa — so the recommendation isn't anchored to one geography/climate.
3. **Difference maps** — pixel-level disagreement between composites
   (F2−F1C and F3−F2) on the unusual-flood signal.

Every plot/table reports the **% of pixels per category** (no water / surface
water / recurring flood / unusual flood / insufficient data).

- **F1C** — 1-day with cloud-shadow screening (sharpest, most noise)
- **F2**  — 2-day max-composite (recommended default in the User Guide)
- **F3**  — 3-day max-composite (smoothest, least noise)

## 0. Environment

Resolve the repo root (so `atlantis` is importable and data paths are stable)
and set a compact plotting style.


In [ ]:
from __future__ import annotations

import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap

# Resolve repo root from the notebook location
NOTEBOOK_DIR = Path.cwd()
candidate = NOTEBOOK_DIR
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
REPO_ROOT = candidate
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "font.size": 9})

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"notebook  = {NOTEBOOK_DIR.relative_to(REPO_ROOT)}")

## 1. Event catalogue & composite config

The five AOI/date windows mirror the `example-*-modis` pixi tasks (see
`pixi.toml`) and [`CLI_Examples.md`](../../../CLI_Examples.md), so this notebook
reuses the exact same inputs the rest of the project benchmarks against.

| Event | bbox `W S E N` | Date window |
|-------|----------------|-------------|
| Valencia 2024 (Spain)        | `-1.5 38.8 0.5 40.0`        | 2024‑10‑29 → 2024‑11‑04 |
| Hurricane Harvey 2017 (USA)  | `-97.27 28.24 -95.54 29.80` | 2017‑08‑28 → 2017‑08‑31 |
| Bihar / Nepal monsoon 2019   | `84.84 24.92 86.49 26.16`   | 2019‑09‑16 → 2019‑09‑20 |
| Typhoon Vamco 2020 (PH)      | `121.14 16.72 122.25 18.45` | 2020‑11‑12 → 2020‑11‑14 |
| West Africa 2020 (GH/TG/BJ)  | `-0.86 8.26 1.99 11.73`     | 2020‑10‑13 → 2020‑10‑15 |

We keep the comparison to the **raw MCDWD pixel codes** (`--no-classify`) so the
composites are judged on their native `0/1/2/3/255` output, not on derived
fractions. `--keep-processed` retains the 250 m `processed/*_modis_raw.tif`
next to the 1‑arcmin `harmonised/*_modis_harmonised.tif` so both resolutions
are available for the same run.


In [ ]:
# Event catalogue — bbox "W S E N" + date window (mirrors pixi.toml example-*-modis tasks)
EVENTS = {
    "Valencia_2024":   {"bbox": "-1.5 38.8 0.5 40.0",        "start": "2024-10-29", "end": "2024-11-04"},
    "Harvey_2017":     {"bbox": "-97.27 28.24 -95.54 29.80", "start": "2017-08-28", "end": "2017-08-31"},
    "Bihar_2019":      {"bbox": "84.84 24.92 86.49 26.16",   "start": "2019-09-16", "end": "2019-09-20"},
    "Vamco_2020":      {"bbox": "121.14 16.72 122.25 18.45", "start": "2020-11-12", "end": "2020-11-14"},
    "WestAfrica_2020": {"bbox": "-0.86 8.26 1.99 11.73",     "start": "2020-10-13", "end": "2020-10-15"},
}

# Composites under test
COMPOSITES = {
    "F1C": "1-day cloud-shadow screened",
    "F2":  "2-day max-composite",
    "F3":  "3-day max-composite",
}

# Canonical MCDWD pixel codes (Release 1.1) — mirrors atlantis.fetchers.modis.layers
MOD_CODE = {
    0:   "No water",
    1:   "Surface water",
    2:   "Recurring flood",
    3:   "Unusual flood",
    255: "Insufficient data",
}
ORDER = [0, 1, 2, 3, 255]
# Readable palette (surface/recurring/unusual kept semantically coloured)
COLORS = {0: "#d4c5a9", 1: "#2980b9", 2: "#f39c12", 3: "#e74c3c", 255: "#7f8c8d"}

DATA_DIR = REPO_ROOT / "notebooks" / "drafts" / "data" / "modis_composites"

print(f"Events    : {len(EVENTS)}  -> {', '.join(EVENTS)}")
print(f"Composites: {len(COMPOSITES)} -> {', '.join(COMPOSITES)}")
print(f"Output root: {DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Total fetches needed: {len(EVENTS) * len(COMPOSITES)} (event x composite)")


## 2. Fetch MODIS for every event × composite

One `atlantis fetch` call per (event, composite). The run is **idempotent**:
if both the 250 m `processed/*_modis_raw.tif` and the 1‑arcmin
`harmonised/*_modis_harmonised.tif` already exist, the cell skips that
combination.

Flags of note:
- `--no-classify` — keep raw `0/1/2/3/255` codes (we judge composites, not derivations)
- `--keep-processed` — retain the 250 m raw so we can compare resolutions
- `--harmonise` — also write the 1‑arcmin grid‑snapped raster
- `--strategy peak` — one peak‑flood layer per composite

> **Prerequisite:** `EARTHDATA_TOKEN` must be set in `.env` at the repo root and
> the LAADS DAAC license for MCDWD_L3 must be accepted on your Earthdata account
> (see [`docs/setup.md`](../../../docs/setup.md)). Each fetch downloads HDF4
> tiles from LAADS, so the first run is network‑bound.


In [ ]:
for event_id, cfg in EVENTS.items():
    for comp, desc in COMPOSITES.items():
        out_dir = DATA_DIR / event_id / comp
        proc = sorted((out_dir / "modis" / "processed").glob("*_modis_raw.tif"))
        harm = sorted((out_dir / "modis" / "harmonised").glob("*_modis_harmonised.tif"))
        if proc and harm:
            print(f"[{event_id:>16}/{comp}] cached -> {proc[0].name}")
            continue

        cmd = [
            "python", "-m", "atlantis.cli", "fetch",
            "--event", event_id,
            "--source", "modis",
            "--bbox", cfg["bbox"],
            "--start-date", cfg["start"],
            "--end-date", cfg["end"],
            "--modis-backend", "laads_hdf4",
            "--modis-composite", comp,
            "--strategy", "peak",
            "--no-classify",
            "--harmonise",
            "--keep-processed",
            "--output", str(out_dir),
        ]
        print(f"[{event_id:>16}/{comp}] fetching ({desc}) ...")
        subprocess.run(
            cmd, cwd=REPO_ROOT, check=True,
            env={**os.environ, "PYTHONPATH": str(SRC)},
        )
print("\nDone.")


## 3. Load both resolutions and tabulate category shares

For every (event, composite) we load:

- **processed** — the 250 m `*_modis_raw.tif` (native MCDWD codes, AOI‑clipped)
- **harmonised** — the 1‑arcmin `*_modis_harmonised.tif` (nearest‑neighbour
  reprojected onto the canonical global grid — *not* majority/mode; see
  `atlantis.fetchers.modis.layers` where the `raw` native layer declares
  `resampling="nearest"`)

The peak date is parsed from the filename with a regex so event IDs containing
underscores (e.g. `Valencia_2024`) are handled correctly.


In [ ]:
DATE_RE = re.compile(r"(\d{4})-?(\d{2})-?(\d{2})")


def peak_date_from_path(path: Path) -> str:
    """Extract the peak date (YYYY-MM-DD) from an output filename."""
    m = DATE_RE.search(path.stem)
    if not m:
        return "unknown"
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"


def category_percent(da: xr.DataArray) -> dict[int, float]:
    """Percentage of pixels in each MCDWD code (255 included — cloud obstruction matters)."""
    vals = da.values.ravel()
    total = vals.size
    return {c: 100.0 * int((vals == c).sum()) / total for c in ORDER}


# loaded[event_id][comp] = {"processed": DataArray, "harmonised": DataArray, "pdate": str}
loaded: dict[str, dict[str, dict]] = {}
rows = []

for event_id in EVENTS:
    loaded[event_id] = {}
    for comp in COMPOSITES:
        out_dir = DATA_DIR / event_id / comp
        proc_tifs = sorted((out_dir / "modis" / "processed").glob("*_modis_raw.tif"))
        harm_tifs = sorted((out_dir / "modis" / "harmonised").glob("*_modis_harmonised.tif"))
        if not proc_tifs or not harm_tifs:
            print(f"[{event_id}/{comp}] MISSING outputs — skipping (run the fetch cell)")
            continue

        # `--keep-processed` retains one 250 m raster per day in the fetch window,
        # but `--harmonise --strategy peak` only harmonises the single peak day.
        # Match the processed file to that same date, otherwise we'd compare two
        # different days (confounding date with resolution).
        pdate = peak_date_from_path(harm_tifs[0])
        proc_match = [p for p in proc_tifs if peak_date_from_path(p) == pdate]
        if not proc_match:
            print(f"[{event_id}/{comp}] no processed raster for peak date {pdate} — skipping")
            continue

        pda = rxr.open_rasterio(proc_match[0]).squeeze(drop=True)
        hda = rxr.open_rasterio(harm_tifs[0]).squeeze(drop=True)
        loaded[event_id][comp] = {"processed": pda, "harmonised": hda, "pdate": pdate}

        for res, da in (("processed", pda), ("harmonised", hda)):
            pct = category_percent(da)
            rows.append({
                "event": event_id, "composite": comp, "resolution": res,
                "peak": pdate, "shape": "x".join(map(str, da.shape)),
                "no_water%": round(pct[0], 2), "surface%": round(pct[1], 2),
                "recurring%": round(pct[2], 2), "unusual%": round(pct[3], 2),
                "insufficient%": round(pct[255], 2),
            })

stats = pd.DataFrame(rows)
n_ok = sum(1 for e in loaded for c in loaded[e])
print(f"Loaded {n_ok}/{len(EVENTS) * len(COMPOSITES)} (event, composite) pairs.")
display(stats)

## 4. Per-event panels: non-harmonised vs harmonised × 3 composites

For each of the five events we draw a **2 × 3** grid:

- **row 0** — 250 m `processed/` (non-harmonised)
- **row 1** — 1-arcmin `harmonised/`

Columns are the three composites, annotated with the **% of pixels per
category**. This is where you can eyeball (a) how compositing changes the
flood signal and (b) how nearest-neighbour down-sampling reshuffles sparse
codes (quantified in §4b below).

In [ ]:
CMAP = ListedColormap([COLORS[c] for c in ORDER])
NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 255.5], CMAP.N)


def _pct_text(da: xr.DataArray) -> str:
    pct = category_percent(da)
    short = {0: "noW", 1: "surf", 2: "recur", 3: "flood", 255: "nodata"}
    return "\n".join(f"{short[c]:>7}: {pct[c]:5.1f}%" for c in ORDER)


def plot_event_panels(event_id: str) -> None:
    """2x3 panel: rows = processed/harmonised, cols = F1C/F2/F3, annotated with % shares."""
    if not loaded.get(event_id):
        print(f"[{event_id}] no data loaded")
        return

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    res_keys = [("processed", "250 m (non-harmonised)"), ("harmonised", "1 arcmin (harmonised)")]

    for r, (res_key, res_label) in enumerate(res_keys):
        for c, comp in enumerate(COMPOSITES):
            ax = axes[r, c]
            entry = loaded[event_id].get(comp)
            if entry is None:
                ax.set_title(f"{comp} — missing", fontsize=9)
                ax.axis("off")
                continue
            da = entry[res_key]
            da.plot.imshow(
                ax=ax, cmap=CMAP, norm=NORM, add_colorbar=False, interpolation="nearest",
            )
            ax.set_title(
                f"{comp}  {res_label}\npeak {entry['pdate']}  ({da.shape[1]}x{da.shape[0]})",
                fontsize=9, fontweight="bold",
            )
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.text(
                0.015, 0.985, _pct_text(da), transform=ax.transAxes,
                va="top", ha="left", fontsize=6.5, family="monospace",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85),
            )

    # Shared legend
    patches = [mpatches.Patch(color=COLORS[c], label=f"{c}: {MOD_CODE[c]}") for c in ORDER]
    fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=8, frameon=True, edgecolor="gray")
    fig.suptitle(f"{event_id} — MODIS composites: non-harmonised vs harmonised", fontsize=12, fontweight="bold")
    plt.tight_layout(rect=[0, 0.06, 1, 0.96])
    plt.show()


for event_id in EVENTS:
    plot_event_panels(event_id)

### 4b. Why do the harmonised rasters sometimes look different?

`--keep-processed` retains one 250 m raster per day in the fetch window, but
`--harmonise --strategy peak` only harmonises the single peak day. §3 now
matches the processed file to that same peak date — for most events the peak
isn't the first day, so an earlier version of this notebook was comparing two
*different calendar dates* (e.g. Harvey's processed was 2017-08-28, >99%
cloud-obscured, against a harmonised 2017-08-30, <8% obscured). That looked
like a dramatic resampling artefact; it was mostly a date mismatch.

With dates matched, the cell below quantifies the *real* resampling effect as
a **retention ratio** — harmonised unusual-flood % ÷ processed unusual-flood
% — per event × composite. Retention ≈ 1.0 means the signal survived; well
below 0.5 means it was thinned by the harmoniser's **nearest-neighbour**
resampling (see `atlantis.fetchers.modis.layers`, `raw` layer,

`resampling="nearest"`), which keeps one 250 m cell per ~7×7 target block anddiscards the rest.

In [ ]:
# Retention ratio = harmonised unusual-flood % / processed unusual-flood %, per event x composite
# (reuses the `stats` table computed above — no need to re-read the rasters).
unusual_piv = (
    stats.pivot_table(index=["event", "composite"], columns="resolution", values="unusual%")
    .rename(columns={"processed": "proc_unusual%", "harmonised": "harm_unusual%"})
    .reset_index()
)
unusual_piv["retention"] = unusual_piv["harm_unusual%"] / unusual_piv["proc_unusual%"].replace(0, np.nan)

# AOI area (deg^2) before/after grid-snapping, from raster bounds
area_rows = []
for event_id in EVENTS:
    for comp in COMPOSITES:
        e = loaded.get(event_id, {}).get(comp)
        if e is None:
            continue
        pw, ps, pe, pn = e["processed"].rio.bounds()
        hw, hs, he_, hn = e["harmonised"].rio.bounds()
        p_area, h_area = (pe - pw) * (pn - ps), (he_ - hw) * (hn - hs)
        area_rows.append({"event": event_id, "composite": comp, "area_ratio": round(h_area / p_area, 3)})

diag = unusual_piv.merge(pd.DataFrame(area_rows), on=["event", "composite"]).round(3)
print("Retention ratio (harmonised/processed unusual-flood %) and AOI area change per event x composite:")
display(diag)

# Mean retention per event (averaged over composites)
mean_ret = diag.groupby("event")["retention"].mean().reindex(EVENTS)
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#7f8c8d" if pd.isna(v) else ("#e74c3c" if v < 0.5 else "#27ae60") for v in mean_ret]
ax.bar(range(len(mean_ret)), mean_ret.values, color=colors, edgecolor="black", linewidth=0.4)
ax.set_xticks(range(len(mean_ret)))
ax.set_xticklabels([s.replace("_", "\n") for s in mean_ret.index], fontsize=8)
ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, label="1:1 (no loss)")
ax.axhline(0.5, color="#e74c3c", linestyle=":", linewidth=0.7, label="0.5 (half lost)")
ax.set_ylabel("retention (harmonised / processed unusual-flood %)")
ax.set_ylim(0, max(1.15, (mean_ret.max() * 1.15) if pd.notna(mean_ret.max()) else 1.15))
ax.set_title(
    "Flood-signal retention under nearest-neighbour harmonisation\n"
    "low = the flood fraction was too sparse for NN resampling to preserve",
    fontsize=10,
)
for i, v in enumerate(mean_ret.values):
    if pd.notna(v):
        ax.text(i, v + 0.03, f"{v:.2f}", ha="center", fontsize=8)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

low = mean_ret[(mean_ret < 0.5) & mean_ret.notna()]
print("Mean retention < 0.5 (signal thinned):", ", ".join(low.index) if not low.empty else "none")

## 5. Difference maps across all events

Where do the composites actually disagree? We compare pairs on the
**unusual‑flood signal (code 3)** using the harmonised rasters (guaranteed
pixel‑aligned because the harmoniser snaps every event to the same canonical
1‑arcmin grid):

- **F2 − F1C** — what does the 2‑day composite remove from the 1‑day result?
  (Red = flood in F1C only → likely cloud‑shadow false‑positive removed by F2.)
- **F3 − F2** — what does the extra day of compositing remove/add?

Pixels masked as insufficient data (255) in *either* composite are excluded
from the comparison. Four classes are drawn: agree‑flood, agree‑no‑flood,
A‑only, B‑only, plus the masked area in white.

> **Caveat:** each composite is shown at its own peak‑flood date (selected by
> max flood‑pixel count). If the peak dates differ for an event, part of the
> disagreement reflects timing rather than the compositing rule — the cell
> warns when this happens.


In [ ]:
DIFF_COLORS = ["#cfcfcf", "#1b5e20", "#e74c3c", "#2980b9"]  # agree-no, agree-yes, A-only, B-only


def _align(a: xr.DataArray, b: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
    """Return a, b on a common grid (reproject_match fallback if shapes differ)."""
    if a.shape == b.shape:
        return a, b
    return a, b.rio.reproject_match(a)


def plot_diff(event_id: str, comp_a: str, comp_b: str) -> None:
    """Categorical disagreement map for the unusual-flood (code 3) signal."""
    ea, eb = loaded.get(event_id, {}).get(comp_a), loaded.get(event_id, {}).get(comp_b)
    if ea is None or eb is None:
        print(f"[{event_id}] missing {comp_a} or {comp_b} — skipping")
        return

    da_a, da_b = _align(ea["harmonised"], eb["harmonised"])
    if ea["pdate"] != eb["pdate"]:
        print(f"  ! [{event_id}] peak dates differ: {comp_a}={ea['pdate']} vs {comp_b}={eb['pdate']} "
              f"— disagreement may partly reflect timing, not just the compositing rule.")

    flood_a = da_a.values == 3
    flood_b = da_b.values == 3
    valid = (da_a.values != 255) & (da_b.values != 255)

    diff = np.full(flood_a.shape, -1, dtype="int8")  # -1 = masked
    diff[valid & ~flood_a & ~flood_b] = 0           # agree: no flood
    diff[valid & flood_a & flood_b] = 1             # agree: flood
    diff[valid & flood_a & ~flood_b] = 2            # A only
    diff[valid & ~flood_a & flood_b] = 3            # B only

    labels = {
        0: f"Agree: no flood",
        1: f"Agree: flood",
        2: f"{comp_a} only",
        3: f"{comp_b} only",
    }
    total_valid = int(valid.sum())
    shares = {k: 100.0 * int((diff == k).sum()) / total_valid for k in labels} if total_valid else {k: 0.0 for k in labels}

    fig, ax = plt.subplots(figsize=(8, 6))
    cmap = ListedColormap(DIFF_COLORS)
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
    masked = np.ma.masked_where(diff < 0, diff)
    ax.imshow(masked, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_title(
        f"{event_id}: {comp_b} minus {comp_a} on unusual flood (code 3)\n"
        f"valid px {total_valid:,}  |  masked {int((~valid).sum()):,}",
        fontsize=10, fontweight="bold",
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    patches = [mpatches.Patch(color=DIFF_COLORS[k], label=f"{labels[k]}: {shares[k]:.1f}%") for k in labels]
    patches.append(mpatches.Patch(color="#ffffff", ec="gray", label="masked (255 in either)"))
    ax.legend(handles=patches, loc="lower center", ncol=2, fontsize=7.5, frameon=True, edgecolor="gray")
    plt.tight_layout()
    plt.show()


# Loop every event over both composite pairs
PAIRS = [("F1C", "F2"), ("F2", "F3")]
for event_id in EVENTS:
    for a, b in PAIRS:
        plot_diff(event_id, a, b)

## 6. Cross-event summary & composite recommendation

Aggregate the harmonised category shares across events, per composite, then
chart the unusual-flood % per event to see how much signal is lost as the
composite window grows (F1C→F2, F2→F3) — the cloud-shadow-suppression vs.
temporal-sharpness trade-off, at a glance.

In [ ]:
# Per-composite means across events (harmonised)
harm_stats = stats[stats.resolution == "harmonised"]
summary = (
    harm_stats.groupby("composite")[["surface%", "recurring%", "unusual%", "insufficient%"]]
    .mean()
    .reindex(list(COMPOSITES))
    .round(2)
)
print("Mean category shares across events (harmonised):")
display(summary)

# Unusual-flood % per event x composite — chart shows the F1C->F2->F3 trend directly
unusual = (
    harm_stats.pivot(index="event", columns="composite", values="unusual%")
    .reindex(index=list(EVENTS), columns=list(COMPOSITES))
)

fig, ax = plt.subplots(figsize=(9, 4))
unusual.plot(kind="bar", ax=ax, rot=0, color=["#c0392b", "#e67e22", "#2980b9"], edgecolor="black", linewidth=0.4)
ax.set_xticklabels([s.replace("_", "\n") for s in unusual.index], fontsize=8)
ax.set_xlabel("")
ax.set_ylabel("unusual-flood % (harmonised)")
ax.set_title("Unusual-flood signal by composite, per event", fontsize=10)
ax.legend(title="composite", fontsize=8)
plt.tight_layout()
plt.show()

# Mean flood-signal loss as the composite window grows (percentage points)
mean_f1c_f2 = (unusual["F1C"] - unusual["F2"]).mean()
mean_f2_f3 = (unusual["F2"] - unusual["F3"]).mean()
mean_insuf = summary["insufficient%"]
print(f"Mean unusual-flood loss  F1C->F2: {mean_f1c_f2:.2f} pp | F2->F3: {mean_f2_f3:.2f} pp")
print("Mean insufficient-data%:", ", ".join(f"{c}={mean_insuf[c]:.2f}" for c in COMPOSITES))

# Recommendation
rec_comp = "F2"
rationale = (
    "F2 keeps most of the flood signal (small F1C->F2 loss) while removing the "
    "bulk of 1-day cloud-shadow noise (the F2->F3 step removes little extra)."
)
if mean_f2_f3 > 0.4 * mean_f1c_f2 and mean_f2_f3 > 0.15:
    rec_comp = "F3"
    rationale = (
        "F3 removes a meaningful extra slice of flood signal beyond F2, suggesting "
        "persistent cloud-shadow noise in F2 for these events — prefer F3."
    )
elif mean_insuf["F1C"] < 25 and mean_f1c_f2 < 0.15:
    rec_comp = "F1C"
    rationale = (
        "F1C already has low cloud obstruction and loses almost nothing to F2, so "
        "the sharper 1-day footprint wins."
    )
print(f"\n>> Recommendation for Atlantis MODIS default: {rec_comp}\n   {rationale}")

## 7. Interpretation — when to use which composite

- **F1C vs F2** — F1C is sharpest but keeps cloud-shadow false positives
  (1-day compositing can't filter transient shadows). F2 requires ≥2
  detections over 2 days, filtering most of them out — expect F1C to show
  more unusual-flood pixels, some of which are noise (visible as "F1C only"
  in the F2−F1C diff map).
- **F2 vs F3** — same logic over 3 days: further noise suppression at the
  cost of some short-duration flood signal. The F2−F3 diff map quantifies
  exactly how much that extra day costs, event by event.
- **Non-harmonised vs harmonised** — see §4b: once compared on the same
  date, nearest-neighbour resampling to 1-arcmin preserves the unusual-flood
  % closely for most events (retention 0.84-1.02). It only breaks down when
  the flood fraction is tiny to begin with (Vamco's F2 composite, ≤0.06% of
  pixels, retention 0.17). This is a harmoniser property, not a composite
  choice — for very sparse flood extents, prefer the classified
  `flood_fraction` layer (resampled with `average`, sub-pixel, sparse-safe)
  over the raw layer's `nearest`.
- **Insufficient-data (255)** — in this run, mean insufficient-data% doesn't
  drop monotonically with longer compositing (F1C 57.3%, F2 56.2%, F3 61.0%
  — printed in §6). That's because each composite's peak day is picked
  independently (the day with most flood signal, not the clearest day), so
  unlike §4b's same-date resolution comparison, this is not an apples-to-apples
  comparison of cloud gaps on a fixed date.
  
### Decision rule

| Composite | Best for |
|-----------|----------|
| **F1C** | Rapid response, single-day event characterisation (accept some noise); best when cloud obstruction is low |
| **F2**  | Event-scale flood mapping — **recommended default per the MCDWD User Guide** (balanced noise vs. temporal sharpness) |
| **F3**  | Cloudy regions / multi-day events where F2 still carries too much cloud-shadow noise |

§6 prints a data-driven pick based on the mean flood-signal loss between
composite steps and the mean cloud obstruction. For these five events, mean
insufficient-data% is high on every composite (~56-61%) — the table's own
"cloudy regions" case — so §6 currently lands on F3 rather than the
general-purpose F2 default above. Swap in your own events and this pick can
change; treat both as a starting point, not a verdict.  

composite steps and the mean cloud obstruction — treat it as a startingpoint, not a verdict.